# CSPF — stage G: validating the automatic fidelity measures on the independent annotator-3 labels
Answers the reviewer's point that the automatic measures (Φ / M1, NLI) were validated only against the two first annotators.
- **G0** quality control of the filled annotator-3 sheet (allowed values, logical consistency, label distribution by family).
- **G1** association of M1 (MiniLM cosine), Φ (= M1 ≥ 0.80 in the audited data) and the bidirectional NLI score with annotator-3 meaning labels: AUC with review-cluster bootstrap CI, for all 325 pairs, the 192 independent pairs and the 133 disputed pairs; pass-rate / precision table for the thresholds used in the paper.
- **G2** flip rate (mean over the 16 classifiers) by annotator-3 meaning label, with cluster-bootstrap CI.
CPU is enough (≈ 650 NLI forward passes). Needs `annotation_sheet_for_third_annotator_FILLED.csv`, `annotator3_KEY_do_not_share.csv` and `ALL_MODELS_combined_FIXED.csv`. Output: `CSPF_stageG_bundle.zip` — send it to me.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
A3_FILLED = f"{BASE}/annotation_sheet_for_third_annotator_FILLED.csv"; A3_KEY = f"{BASE}/annotator3_KEY_do_not_share.csv"; FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
# if a small file is not in the Drive folder, a file-upload dialog opens (run-all friendly: just pick the file when asked)
def ensure(path):
    import os
    if os.path.exists(path): return path
    from google.colab import files
    print("not found:", path, "-> please upload", os.path.basename(path)); up = files.upload(); name = list(up)[0]; os.replace(name, "/content/"+os.path.basename(path)); return "/content/"+os.path.basename(path)
A3_FILLED = ensure(A3_FILLED); A3_KEY = ensure(A3_KEY)
OUT = f"{BASE}/results_FIXED/reviewer_G"; B_REP = 1000
import os, subprocess, sys, warnings, zipfile, glob; warnings.filterwarnings("ignore"); os.makedirs(OUT, exist_ok=True)
subprocess.run([sys.executable,"-m","pip","-q","install","sentence-transformers"],check=False)
import numpy as np, pandas as pd
from scipy.stats import spearmanr
rng = np.random.default_rng(0)

In [ ]:
# ── G0: load, validate, describe ──
a = pd.read_csv(A3_FILLED, encoding="utf-8-sig"); k = pd.read_csv(A3_KEY, encoding="utf-8-sig")
for c in ["meaning_a3","sentiment_a3","natural_a3"]: a[c] = a[c].astype(str).str.strip().str.lower()
ALLOWED = {"meaning_a3":{"yes","partial","no"}, "sentiment_a3":{"yes","unclear","no"}, "natural_a3":{"yes","borderline","no"}}
for c, s in ALLOWED.items(): print(c, "invalid values:", sorted(set(a[c]) - s), "| missing:", int(a[c].isin(["nan",""]).sum()))
d = a.merge(k[["sample_id","type","perturbation_family","severity","source_dataset","review_idx"]], on="sample_id", how="inner"); assert len(d) == len(a) == len(k), (len(d), len(a), len(k))
d["key"] = d.source_dataset.astype(str)+"::"+d.review_idx.astype(str)
print("\nlabel distribution (share):"); print(pd.concat({c: d[c].value_counts(normalize=True).round(3) for c in ALLOWED}, axis=1).fillna(0))
print("\nlogical consistency: meaning=yes but sentiment!=yes:", int(((d.meaning_a3=="yes") & (d.sentiment_a3!="yes")).sum()), "| sentiment=no but meaning=yes:", int(((d.sentiment_a3=="no") & (d.meaning_a3=="yes")).sum()))
fam = d.assign(yes=(d.meaning_a3=="yes"), no=(d.meaning_a3=="no"), nat=(d.natural_a3=="yes")).groupby("perturbation_family").agg(n=("yes","size"), meaning_yes=("yes","mean"), meaning_no=("no","mean"), natural_yes=("nat","mean")).round(2)
print("\nby family:"); print(fam); fam.to_csv(f"{OUT}/G0_label_by_family.csv"); d.groupby("type").meaning_a3.value_counts(normalize=True).round(3).to_csv(f"{OUT}/G0_label_by_type.csv")

In [ ]:
# ── G1: automatic measures vs annotator-3 meaning labels ──
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device:", dev)
def embed_cos(orig, pert):
    m = SentenceTransformer("all-MiniLM-L6-v2", device=dev)
    A = m.encode(list(orig), normalize_embeddings=True, batch_size=64); B = m.encode(list(pert), normalize_embeddings=True, batch_size=64); return (A*B).sum(1)
@torch.no_grad()
def nli_bi(orig, pert, bs=32, name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"):
    t = AutoTokenizer.from_pretrained(name); m = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
    e = {v.lower(): kk for kk, v in m.config.id2label.items()}["entailment"]
    def run(P, H):
        out = np.zeros(len(P), dtype=np.float32)
        for i in range(0, len(P), bs):
            enc = t(P[i:i+bs], H[i:i+bs], padding=True, truncation=True, max_length=160, return_tensors="pt").to(dev); out[i:i+bs] = torch.softmax(m(**enc).logits.float(), -1)[:, e].cpu().numpy()
        return out
    return np.minimum(run(list(orig), list(pert)), run(list(pert), list(orig)))

In [ ]:
d["m1"] = embed_cos(d.original_review, d.perturbed_review); d["nli_bi"] = nli_bi(d.original_review, d.perturbed_review)
d["phi_pass"] = (d.m1 >= 0.80).astype(int)          # Phi reduces to the M1 >= 0.80 test in the audited data (Appendix A.6)
d["y_yes"] = (d.meaning_a3 == "yes").astype(int); d["y_no"] = (d.meaning_a3 == "no").astype(int); d["ord"] = d.meaning_a3.map({"no":0,"partial":1,"yes":2})
from sklearn.metrics import roc_auc_score
def auc(y, s):
    return np.nan if len(np.unique(y)) < 2 else roc_auc_score(y, s)
def cboot(sub, fn, B=B_REP):
    uk, ki = np.unique(sub.key.values, return_inverse=True); idx_by = [np.where(ki == i)[0] for i in range(len(uk))]; v = []
    for _ in range(B):
        ix = np.concatenate([idx_by[i] for i in rng.integers(0, len(uk), len(uk))]); r = fn(sub.iloc[ix])
        if not np.isnan(r): v.append(r)
    return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 50 else (np.nan, np.nan)
subsets = {"all 325": d, "independent 192": d[d.type=="new_independent"], "disputed 133": d[d.type=="disputed"]}
scores = {"M1 cosine (= Phi score)": "m1", "NLI bidirectional": "nli_bi"}; rows = []
for sn, sub in subsets.items():
    for mn, col in scores.items():
        for tn, tgt, sgn in [("meaning-preserving (yes vs rest)", "y_yes", 1), ("meaning-altering (no vs rest)", "y_no", -1)]:
            f = lambda s, col=col, tgt=tgt, sgn=sgn: auc(s[tgt].values, sgn*s[col].values); est = f(sub); lo, hi = cboot(sub, f)
            rows.append(dict(subset=sn, measure=mn, target=tn, n=len(sub), n_pos=int(sub[tgt].sum()), AUC=round(est,3), ci_lo=round(lo,3), ci_hi=round(hi,3)))
        rows.append(dict(subset=sn, measure=mn, target="Spearman with ordinal label", n=len(sub), n_pos=np.nan, AUC=round(spearmanr(sub[col], sub.ord)[0],3), ci_lo=np.nan, ci_hi=np.nan))
G1 = pd.DataFrame(rows); print(G1.to_string(index=False)); G1.to_csv(f"{OUT}/G1_auc.csv", index=False)
rows = []
for sn, sub in subsets.items():
    for rule, m in [("Phi pass (M1 >= 0.80)", sub.phi_pass == 1), ("NLI-bi >= 0.5", sub.nli_bi >= .5), ("NLI-bi >= 0.9", sub.nli_bi >= .9)]:
        rows.append(dict(subset=sn, rule=rule, share_pass=round(m.mean(),3), P_yes_given_pass=round(sub.y_yes[m].mean(),3) if m.any() else np.nan, P_no_given_pass=round(sub.y_no[m].mean(),3) if m.any() else np.nan, recall_of_yes=round(m[sub.y_yes==1].mean(),3) if sub.y_yes.sum() else np.nan))
G1b = pd.DataFrame(rows); print(G1b.to_string(index=False)); G1b.to_csv(f"{OUT}/G1_threshold_rules.csv", index=False); d.drop(columns=["original_review","perturbed_review","comment_a3"], errors="ignore").to_csv(f"{OUT}/G1_scores_per_pair.csv", index=False)

In [ ]:
# ── G2: flip rate by annotator-3 meaning label (skipped, with a message, if the big prediction file is not in the Drive folder) ──
HAVE_FIXED = os.path.exists(FIXED_CSV); print("prediction file found:", HAVE_FIXED)
if HAVE_FIXED:
  hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist(); MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
  meta = ["source_dataset","review_idx","perturbed_review","perturbation_family"]; use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
  dd = d.copy(); dd["source_dataset"] = dd.source_dataset.astype(str); dd["review_idx"] = dd.review_idx.astype(str); kk = dd.set_index(meta); parts = []
  for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
      ch["source_dataset"] = ch.source_dataset.astype(str); ch["review_idx"] = ch.review_idx.astype(str)
      hit = ch.merge(dd[meta+["sample_id"]], on=meta, how="inner")
      if len(hit): parts.append(hit)
  H = pd.concat(parts).drop_duplicates("sample_id"); print(len(H), "of", len(d), "pairs matched to the prediction file")
  Fm = np.stack([(H[f"{m}_orig_label"].values != H[f"{m}_pert_label"].values) for m in MODELS], 1).astype(float); H["y"] = Fm.mean(1)
  J = H[["sample_id","y"]].merge(d[["sample_id","meaning_a3","type","key"]], on="sample_id"); rows = []
  for nm, sub in [("all matched", J)] + [(f"a3: {l}", J[J.meaning_a3==l]) for l in ["yes","partial","no"]] + [("a3: yes, independent only", J[(J.meaning_a3=="yes")&(J.type=="new_independent")])]:
      if len(sub) == 0: continue
      lo, hi = cboot(sub, lambda s: s.y.mean()); rows.append(dict(group=nm, pairs=len(sub), FR=round(sub.y.mean(),4), ci_lo=round(lo,4), ci_hi=round(hi,4)))
  G2 = pd.DataFrame(rows); print(G2.to_string(index=False)); G2.to_csv(f"{OUT}/G2_FR_by_a3_label.csv", index=False)

with zipfile.ZipFile(f"{OUT}/CSPF_stageG_bundle.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv"): z.write(f, os.path.basename(f))
print("send me:", f"{OUT}/CSPF_stageG_bundle.zip")
try:
    from google.colab import files; files.download(f"{OUT}/CSPF_stageG_bundle.zip")
except Exception as e: print("download manually from Drive:", e)